# NYC Taxi Trip Quality and Fare Anomaly Analysis

**Dataset:** Databricks `samples.nyctaxi.trips` (2016 sample records)  
**Tools:** Databricks SQL, SQL CTEs, aggregation, filtering, derived speed and fare-per-mile metrics  
**Scope:** Exploratory anomaly review, **not** fraud classification.

This notebook documents data-quality checks, filtering, descriptive statistics, implausible travel speeds, fare-per-mile patterns, and a repeated 52-dollar fare. Results below were exported from Databricks; to reproduce them, use a Databricks workspace with access to the sample table.


In [ ]:
%sql
-- Data-quality overview of the Databricks sample NYC taxi table.
-- Review these counts before making choices about which trips to analyze.
SELECT
    COUNT(*) AS total_trips,
    SUM(CASE WHEN tpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) AS missing_pickup_time,
    SUM(CASE WHEN tpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS missing_dropoff_time,
    SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END) AS missing_distance,
    SUM(CASE WHEN fare_amount IS NULL THEN 1 ELSE 0 END) AS missing_fare,
    SUM(CASE WHEN trip_distance <= 0 THEN 1 ELSE 0 END) AS nonpositive_distance,
    SUM(CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END) AS negative_fare,
    SUM(CASE
            WHEN tpep_pickup_datetime IS NOT NULL
             AND tpep_dropoff_datetime IS NOT NULL
             AND tpep_dropoff_datetime <= tpep_pickup_datetime
            THEN 1 ELSE 0
        END) AS nonpositive_duration
FROM samples.nyctaxi.trips;



total_trips,missing_pickup_time,missing_dropoff_time,missing_distance,missing_fare,nonpositive_distance,negative_fare,nonpositive_duration
21932,0,0,0,0,76,5,1


## Data Quality Findings

The dataset contains 21,932 NYC taxi trips.

### Results
- No missing values were found in pickup time, drop-off time, trip distance, or fare amount.
- 76 trips had zero or negative distances.
- 5 trips had negative fares.
- 1 trip had a zero or negative duration.

### Interpretation
Although the dataset has no missing values in the fields checked, some records contain unusual values that may affect the analysis.

These records should be investigated before deciding whether to exclude them.

### Next Steps
Examine trips with unusual distances, fares, or durations before analyzing overall fare patterns.


In [ ]:

%sql
-- Examine taxi trips with unusual distances, fares, or durations

SELECT
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    trip_distance,
    fare_amount,
    ROUND(
        (unix_timestamp(tpep_dropoff_datetime)
         - unix_timestamp(tpep_pickup_datetime)) / 60.0,
        2
    ) AS duration_minutes
FROM samples.nyctaxi.trips
WHERE trip_distance <= 0
   OR fare_amount < 0
   OR tpep_dropoff_datetime <= tpep_pickup_datetime
ORDER BY trip_distance ASC, fare_amount ASC
LIMIT 20;


tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,duration_minutes
2016-02-14T22:06:26.000Z,2016-02-14T22:06:46.000Z,0.0,0.0,0.33
2016-02-18T14:20:19.000Z,2016-02-18T14:21:09.000Z,0.0,0.01,0.83
2016-01-07T07:05:55.000Z,2016-01-07T07:06:06.000Z,0.0,2.5,0.18
2016-02-12T18:40:28.000Z,2016-02-12T18:40:50.000Z,0.0,2.5,0.37
2016-02-03T09:43:58.000Z,2016-02-03T09:44:28.000Z,0.0,2.5,0.50
2016-02-13T12:58:30.000Z,2016-02-13T12:59:06.000Z,0.0,2.5,0.60
2016-02-14T21:17:09.000Z,2016-02-14T21:17:34.000Z,0.0,2.5,0.42
2016-02-27T23:54:24.000Z,2016-02-27T23:54:27.000Z,0.0,2.5,0.05
2016-02-04T05:46:02.000Z,2016-02-04T05:46:28.000Z,0.0,2.5,0.43
2016-01-26T19:14:26.000Z,2016-01-26T19:14:46.000Z,0.0,2.5,0.33


## Investigation of Unusual Taxi Trips

### Findings

I examined 20 taxi trips flagged for unusual distance, fare,
or duration values.

- All 20 trips recorded 0 miles traveled.
- Most trips had a fare of $2.50.
- All 20 trips lasted less than one minute.
- One trip lasted only 3 seconds.

### Interpretation

These trips may represent canceled rides, incomplete trips,
or errors in the recorded data.

The repeated $2.50 fare is worth investigating because
passengers were charged even though the recorded distance
was zero.

However, these records alone do not prove fraudulent activity.

### Next Steps

Examine the negative-fare records and compare the flagged
trips with the rest of the dataset before deciding which
records to exclude.


In [ ]:

%sql
-- Investigate taxi trips with negative fares

SELECT
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    trip_distance,
    fare_amount,
    ROUND(
        (unix_timestamp(tpep_dropoff_datetime)
         - unix_timestamp(tpep_pickup_datetime)) / 60.0,
        2
    ) AS duration_minutes
FROM samples.nyctaxi.trips
WHERE fare_amount < 0
ORDER BY fare_amount ASC;


tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,duration_minutes
2016-01-26T14:02:10.000Z,2016-01-26T14:14:16.000Z,0.16,-8.0,12.10
2016-02-04T13:33:17.000Z,2016-02-04T13:39:25.000Z,0.48,-4.5,6.13
2016-02-02T10:48:08.000Z,2016-02-02T10:51:28.000Z,0.46,-4.0,3.33
2016-01-05T09:33:49.000Z,2016-01-05T09:36:25.000Z,0.52,-4.0,2.60
2016-02-20T21:02:39.000Z,2016-02-20T21:03:47.000Z,0.05,-3.0,1.13


## Investigation of Negative Fares

### Findings

The dataset contains 5 trips with negative fares out of
21,932 total trips (approximately 0.023%).

- Negative fares ranged from -$8.00 to -$3.00.
- All five trips recorded positive distances.
- All five trips had positive durations.
- Every trip traveled less than one mile.

### Interpretation

Negative fares are uncommon in this dataset.

These values may represent refunds, payment reversals,
corrections, or data recording errors.

Additional transaction information would be needed to
determine why the fares were negative.

### Data Cleaning Decision

For the next analysis of typical taxi fares, negative-fare
records will be excluded because they do not represent
ordinary positive fare charges.

However, the original records will be preserved for
future investigation.


In [ ]:

%sql
-- Count records suitable for typical fare analysis

SELECT
    COUNT(*) AS valid_trips
FROM samples.nyctaxi.trips
WHERE trip_distance > 0
  AND fare_amount > 0
  AND tpep_dropoff_datetime > tpep_pickup_datetime;


valid_trips
21847


## Data Cleaning Results

### Original Dataset
- Total trips: 21,932

### Cleaned Dataset
- Valid trips: 21,847
- Excluded trips: 85
- Data retained: 99.61%

### Cleaning Criteria
Trips were retained when they had:
- A positive trip distance
- A positive fare amount
- A drop-off time later than the pickup time

### Interpretation
The majority of records met the criteria for analyzing
typical taxi fares.

Only 0.39% of trips were excluded from this analysis.

The original dataset remains unchanged.


In [ ]:

%sql
-- Calculate summary statistics for valid NYC taxi trips

SELECT
    COUNT(*) AS total_valid_trips,
    ROUND(AVG(trip_distance), 2) AS avg_distance_miles,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(MIN(fare_amount), 2) AS min_fare,
    ROUND(MAX(fare_amount), 2) AS max_fare,
    ROUND(AVG(
        (unix_timestamp(tpep_dropoff_datetime)
        - unix_timestamp(tpep_pickup_datetime)) / 60.0
    ), 2) AS avg_duration_minutes
FROM samples.nyctaxi.trips
WHERE trip_distance > 0
  AND fare_amount > 0
  AND tpep_dropoff_datetime > tpep_pickup_datetime;


total_valid_trips,avg_distance_miles,avg_fare,min_fare,max_fare,avg_duration_minutes
21847,2.86,12.31,0.01,275.0,15.16


## Summary Statistics for Cleaned Taxi Data

### Results

After filtering the dataset, 21,847 trips remained.

- Average trip distance: 2.86 miles
- Average fare: $12.31
- Minimum fare: $0.01
- Maximum fare: $275.00
- Average trip duration: 15.16 minutes

### Interpretation

The average taxi trip traveled approximately 2.86 miles,
lasted 15.16 minutes, and cost $12.31.

The maximum fare of $275 is substantially higher than
the average fare and may represent an unusual trip.

The minimum fare of $0.01 is also worth investigating.

These values do not automatically indicate fraud.
Additional information is needed to understand whether
they represent legitimate trips or data anomalies.

### Next Steps

Examine the trips with the highest fares and compare
their distances and durations to determine whether
the fares appear unusual relative to the trip details.


In [ ]:

%sql
-- Examine the 10 highest taxi fares

SELECT
    tpep_pickup_datetime,
    trip_distance,
    fare_amount,
    ROUND(
        (unix_timestamp(tpep_dropoff_datetime)
         - unix_timestamp(tpep_pickup_datetime)) / 60.0,
        2
    ) AS duration_minutes,
    ROUND(
        fare_amount / trip_distance,
        2
    ) AS fare_per_mile
FROM samples.nyctaxi.trips
WHERE trip_distance > 0
  AND fare_amount > 0
  AND tpep_dropoff_datetime > tpep_pickup_datetime
ORDER BY fare_amount DESC
LIMIT 10;


tpep_pickup_datetime,trip_distance,fare_amount,duration_minutes,fare_per_mile
2016-02-12T20:55:19.000Z,20.85,275.0,57.32,13.19
2016-02-17T22:23:14.000Z,25.46,130.0,43.02,5.11
2016-01-28T17:36:17.000Z,21.3,115.0,51.52,5.4
2016-01-04T09:19:53.000Z,5.2,95.0,0.07,18.27
2016-02-24T22:19:55.000Z,12.49,95.0,16.12,7.61
2016-02-22T21:17:27.000Z,30.6,95.0,43.52,3.1
2016-01-21T00:58:34.000Z,24.5,94.5,34.60,3.86
2016-01-22T15:01:35.000Z,22.9,93.5,85.93,4.08
2016-02-11T12:47:12.000Z,19.02,88.0,29.78,4.63
2016-01-25T14:31:21.000Z,17.4,85.5,72.50,4.91


## Investigation of High-Fare Taxi Trips

### Findings

I examined the 10 trips with the highest fares.

- The highest fare was $275 for a 20.85-mile trip.
- Several expensive trips covered relatively long distances.
- One unusual trip recorded a $95 fare for 5.2 miles.
- This $95 trip had a recorded duration of approximately
  4 seconds.

### Interpretation

The $95 trip is particularly unusual because traveling
5.2 miles in approximately 4 seconds would require
an unrealistic speed.

This suggests a possible timestamp or distance recording
error.

### Important Observation

The original cleaning process required positive distances,
fares, and durations. However, those conditions did not
identify trips with unrealistic implied speeds.

### Next Steps

Calculate the average speed of each trip and identify
records with unrealistic travel speeds.

These records will be flagged for further investigation
rather than automatically classified as fraudulent.


In [ ]:

%sql
-- Identify taxi trips with unusually high implied speeds

WITH trip_speeds AS (
    SELECT
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        trip_distance,
        fare_amount,
        ROUND(
            (unix_timestamp(tpep_dropoff_datetime)
             - unix_timestamp(tpep_pickup_datetime)) / 60.0,
            2
        ) AS duration_minutes,
        ROUND(
            trip_distance /
            ((unix_timestamp(tpep_dropoff_datetime)
              - unix_timestamp(tpep_pickup_datetime)) / 3600.0),
            2
        ) AS avg_speed_mph
    FROM samples.nyctaxi.trips
    WHERE trip_distance > 0
      AND fare_amount > 0
      AND tpep_dropoff_datetime > tpep_pickup_datetime
)

SELECT *
FROM trip_speeds
WHERE avg_speed_mph > 100
ORDER BY avg_speed_mph DESC;


tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,duration_minutes,avg_speed_mph
2016-01-13T05:55:31.000Z,2016-01-13T05:55:34.000Z,8.7,52.0,0.05,10444.18
2016-01-07T19:05:14.000Z,2016-01-07T19:05:17.000Z,5.0,2.5,0.05,6002.4
2016-01-10T03:40:39.000Z,2016-01-10T03:40:41.000Z,3.3,2.5,0.03,5935.25
2016-01-04T09:19:53.000Z,2016-01-04T09:19:57.000Z,5.2,95.0,0.07,4680.47
2016-01-02T18:04:46.000Z,2016-01-02T18:04:58.000Z,15.3,52.0,0.20,4590.46
2016-01-02T12:07:31.000Z,2016-01-02T12:07:36.000Z,5.6,2.5,0.08,4031.68
2016-01-01T00:31:05.000Z,2016-01-01T00:31:08.000Z,2.9,13.5,0.05,3481.39
2016-01-07T09:58:59.000Z,2016-01-07T09:59:04.000Z,2.6,2.5,0.08,1871.85
2016-01-05T13:29:56.000Z,2016-01-05T13:30:23.000Z,11.8,40.0,0.45,1573.33
2016-02-26T10:24:46.000Z,2016-02-26T10:24:52.000Z,2.1,2.5,0.10,1259.75


## Investigation of Unrealistic Travel Speeds

### Findings

After calculating average trip speeds, I identified
23 trips with implied speeds greater than 100 mph.

- The highest calculated speed was approximately 10,444 mph.
- Several trips recorded multiple miles of travel in only
  a few seconds.
- The previously identified $95 trip was among the
  flagged records.
- Some trips with unrealistic speeds had low fares.

### Interpretation

These speeds are unrealistic for NYC taxi trips and
suggest possible timestamp, distance, or recording errors.

The results demonstrate that positive distances and
durations do not necessarily indicate valid trip records.

### Data Cleaning Decision

For the next analysis, I will exclude trips with
calculated average speeds greater than 100 mph.

The original data will remain unchanged.

The 100 mph threshold is an exploratory data-quality rule,
not a validated fraud-detection threshold.


In [ ]:

%sql
-- Refine the cleaned dataset by excluding unrealistic speeds

WITH trip_speeds AS (
    SELECT
        *,
        trip_distance /
        ((unix_timestamp(tpep_dropoff_datetime)
          - unix_timestamp(tpep_pickup_datetime)) / 3600.0)
        AS avg_speed_mph
    FROM samples.nyctaxi.trips
    WHERE trip_distance > 0
      AND fare_amount > 0
      AND tpep_dropoff_datetime > tpep_pickup_datetime
)

SELECT
    COUNT(*) AS final_valid_trips,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(avg_speed_mph), 2) AS avg_speed_mph
FROM trip_speeds
WHERE avg_speed_mph <= 100;


final_valid_trips,avg_distance,avg_fare,avg_speed_mph
21824,2.86,12.31,11.98


## Final Data Cleaning Results

### Dataset Summary

- Original trips: 21,932
- Trips after initial filtering: 21,847
- Trips flagged for unrealistic speeds: 23
- Final retained trips: 21,824
- Total excluded trips: 108
- Percentage retained: 99.51%

### Final Summary Statistics

- Average trip distance: 2.86 miles
- Average fare: $12.31
- Average implied speed: 11.98 mph

### Cleaning Method

I applied the following criteria:

1. Trip distance must be greater than zero.
2. Fare amount must be greater than zero.
3. Drop-off time must be later than pickup time.
4. Implied average speed must be 100 mph or less.

### Interpretation

The additional speed check identified 23 unusual
records that passed the original filtering criteria.

This shows why checking only for missing or negative
values is not always enough to assess data quality.

The 100 mph threshold was selected as an exploratory
filter and should not be interpreted as a validated
fraud-detection rule.

The original dataset remains unchanged.


In [ ]:

%sql
-- Identify trips with unusually high fares per mile
-- Using the refined cleaned dataset

WITH cleaned_trips AS (
    SELECT
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        trip_distance,
        fare_amount,
        (unix_timestamp(tpep_dropoff_datetime)
         - unix_timestamp(tpep_pickup_datetime)) / 3600.0
         AS duration_hours
    FROM samples.nyctaxi.trips
    WHERE trip_distance > 0
      AND fare_amount > 0
      AND tpep_dropoff_datetime > tpep_pickup_datetime
),

fare_analysis AS (
    SELECT
        *,
        ROUND(fare_amount / trip_distance, 2)
            AS fare_per_mile
    FROM cleaned_trips
    WHERE trip_distance / duration_hours <= 100
)

SELECT
    tpep_pickup_datetime,
    trip_distance,
    fare_amount,
    ROUND(duration_hours * 60, 2) AS duration_minutes,
    fare_per_mile
FROM fare_analysis
WHERE trip_distance >= 1
ORDER BY fare_per_mile DESC
LIMIT 10;


tpep_pickup_datetime,trip_distance,fare_amount,duration_minutes,fare_per_mile
2016-01-31T16:32:47.000Z,1.12,52.0,7.38,46.43
2016-02-04T19:09:13.000Z,1.5,39.0,71.80,26.0
2016-01-16T10:24:56.000Z,2.39,52.0,9.00,21.76
2016-01-25T11:20:54.000Z,1.12,23.5,42.53,20.98
2016-01-26T12:10:30.000Z,1.03,20.5,35.95,19.9
2016-02-29T17:59:36.000Z,2.88,52.0,18.13,18.06
2016-01-14T22:08:47.000Z,2.94,52.0,9.77,17.69
2016-01-26T09:04:58.000Z,3.0,52.0,38.28,17.33
2016-02-02T17:41:49.000Z,1.0,17.0,29.08,17.0
2016-02-17T17:58:10.000Z,1.06,17.0,29.25,16.04


## Investigation of High Fares per Mile

### Findings

I examined the 10 trips with the highest fare per mile
among cleaned trips traveling at least one mile.

- The highest fare per mile was $46.43.
- One trip cost $52 for only 1.12 miles.
- Another trip traveled 1.5 miles over 71.80 minutes.
- Four of the 10 trips had fares of exactly $52.

### Interpretation

Some trips had unusually high fares relative to
the recorded distance traveled.

Long trip durations may help explain certain high fares,
while repeated fare amounts could indicate fixed-rate
pricing or other legitimate billing practices.

These results do not establish fraudulent activity.

### Next Steps

Investigate repeated fare amounts and compare the
distribution of fares across the cleaned dataset.

The goal is to distinguish unusual observations from
potentially legitimate pricing patterns.


In [ ]:

%sql
-- Examine repeated fare amounts in the cleaned dataset

WITH cleaned_trips AS (
    SELECT
        trip_distance,
        fare_amount,
        trip_distance /
        ((unix_timestamp(tpep_dropoff_datetime)
        - unix_timestamp(tpep_pickup_datetime)) / 3600.0)
        AS avg_speed_mph
    FROM samples.nyctaxi.trips
    WHERE trip_distance > 0
      AND fare_amount > 0
      AND tpep_dropoff_datetime > tpep_pickup_datetime
)

SELECT
    fare_amount,
    COUNT(*) AS number_of_trips,
    ROUND(AVG(trip_distance), 2) AS avg_distance
FROM cleaned_trips
WHERE avg_speed_mph <= 100
GROUP BY fare_amount
ORDER BY number_of_trips DESC
LIMIT 15;


fare_amount,number_of_trips,avg_distance
5.5,1137,0.84
6.5,1113,1.08
6.0,1089,0.94
7.0,1076,1.19
5.0,1045,0.72
7.5,1022,1.3
8.0,964,1.43
8.5,921,1.56
4.5,867,0.62
9.0,833,1.68


## Distribution of Common Taxi Fares

### Findings

I identified the 15 most frequently occurring fare amounts
in the cleaned NYC taxi dataset.

- The most common fare was $5.50, appearing in 1,137 trips.
- The 15 most common fares ranged from $4.00 to $11.00.
- The average distance generally increased with fare amount.
- Common fare amounts appeared in $0.50 increments.

### Interpretation

The results suggest a relationship between trip distance
and fare amount.

The repeated fare amounts may reflect structured taxi
pricing rather than unusual activity.

The previously observed $52 fare was not among the
15 most frequently occurring fares.

### Next Steps

Investigate the frequency and trip characteristics
associated with the $52 fare.


In [ ]:

%sql
-- Investigate trips with an exact fare of 52 dollars

WITH cleaned_trips AS (
    SELECT
        trip_distance,
        fare_amount,
        (unix_timestamp(tpep_dropoff_datetime)
         - unix_timestamp(tpep_pickup_datetime)) / 3600.0
         AS duration_hours
    FROM samples.nyctaxi.trips
    WHERE trip_distance > 0
      AND fare_amount > 0
      AND tpep_dropoff_datetime > tpep_pickup_datetime
),

valid_trips AS (
    SELECT *
    FROM cleaned_trips
    WHERE trip_distance / duration_hours <= 100
)

SELECT
    COUNT(*) AS trips_with_52_fare,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(MIN(trip_distance), 2) AS min_distance,
    ROUND(MAX(trip_distance), 2) AS max_distance,
    ROUND(AVG(duration_hours * 60), 2) AS avg_duration_minutes
FROM valid_trips
WHERE fare_amount = 52;


trips_with_52_fare,avg_distance,min_distance,max_distance,avg_duration_minutes
402,17.64,0.03,23.45,43.12


## Investigation of Repeated Fare Amounts

### Findings

I investigated taxi trips with an exact fare of $52.

- Number of trips: 402
- Average distance: 17.64 miles
- Minimum distance: 0.03 miles
- Maximum distance: 23.45 miles
- Average duration: 43.12 minutes

### Interpretation

The repeated $52 fare across trips with different
distances suggests that some trips may follow
a fixed-rate pricing structure.

A $52 flat fare was historically associated with
certain NYC airport trips during this period.

However, pickup and drop-off locations would be needed
to verify whether these records represent airport trips.

### Conclusion

An unusually high fare per mile does not necessarily
indicate fraudulent activity.

Legitimate pricing structures and data-recording
issues should be considered when evaluating anomalies.
